In [10]:
df = pd.read_csv("data/S1/BVP.csv", header=None)

print(df.head())
print("Shape:", df.shape)

fs = df.iloc[0, 0]
print("Sampling rate (fs):", fs)

bvp = df.iloc[1:, 0].values
print("Signal length:", len(bvp))
print("Duration (seconds):", len(bvp)/fs)

              0
0  1.530262e+09
1  6.400000e+01
2 -0.000000e+00
3 -0.000000e+00
4 -0.000000e+00
Shape: (597907, 1)
Sampling rate (fs): 1530261892.0
Signal length: 597906
Duration (seconds): 0.00039072135503456687


In [24]:
import os
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt

def bandpass_filter(signal, fs, low=0.7, high=4.0, order=3):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, signal)

base_path = "data"

all_features = []

for subject in os.listdir(base_path):

    print("Processing:", subject)

    bvp_path = os.path.join(base_path, subject, "BVP.csv")

    if not os.path.exists(bvp_path):
        print("  ❌ BVP.csv not found")
        continue

    df = pd.read_csv(bvp_path, header=None)

    fs = 64
    bvp = df.iloc[:, 0].values

    print("  Signal length:", len(bvp))

    filtered = bandpass_filter(bvp, fs)
    filtered = (filtered - np.mean(filtered)) / np.std(filtered)

    min_len = fs * 5

    if len(filtered) < min_len:
        print("  ❌ Too short, skipped")
        continue

# Create multiple segments from S1

for i in range(0, len(filtered) - min_len, min_len):

    segment = filtered[i:i + min_len]

    features = {
        "subject": subject,
        "mean": np.mean(segment),
        "std": np.std(segment),
        "variance": np.var(segment),
        "max": np.max(segment),
        "min": np.min(segment),
        "range": np.max(segment) - np.min(segment),
        "energy": np.sum(segment**2)
    }

    all_features.append(features)
    print("  ✅ Features added")

df_all = pd.DataFrame(all_features)

print("\nTotal samples collected:", len(df_all))
print("Length:", len(filtered), "Required:", min_len)

df_all.to_csv("dataset_features.csv", index=False)

Processing: S1
  Signal length: 597907
Processing: S10
  Signal length: 692397
Processing: S11
  Signal length: 586522
Processing: S12
  Signal length: 509764
Processing: S13
  Signal length: 592946
Processing: S14
  Signal length: 578239
Processing: S15
  Signal length: 515308
Processing: S2
  Signal length: 534228
Processing: S3
  Signal length: 566667
Processing: S4
  Signal length: 589272
Processing: S5
  Signal length: 599722
Processing: S6
  Signal length: 637738
Processing: S7
  Signal length: 603286
Processing: S8
  Signal length: 524240
Processing: S9
  Signal length: 557240
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Features added
  ✅ Featur